In [1]:
from langgraph.graph import StateGraph , START , END
from typing import TypedDict , Annotated
from langchain_core.messages import BaseMessage , HumanMessage
from langgraph.graph.message import add_messages
from dotenv import load_dotenv

from langgraph.prebuilt import ToolNode , tools_condition
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
import requests
import random
from langchain_groq import ChatGroq


In [2]:
llm = ChatGroq(
    api_key = "YOUR_GROQ_API_KEY",
    model = "deepseek-r1-distill-llama-70b"
)

In [3]:
search_tool = DuckDuckGoSearchRun(region = "us-en")

In [4]:
@tool
def calculator(first_num : float , second_num : float , operation:str) -> dict:
    """
    Perform a basic arithmetic operation on two numbers.
    Supported operations : add , sub , mul , div
    """

    try:
        if operation == "add":
            result = first_num + second_num
        elif operation == "sub":
            result = first_num - second_num
        elif operation == "mul":
            result = first_num * second_num
        elif operation == "div":
            if second_num == 0:
                return {"error" : "Division by zero is not allowed"}
            result = first_num / second_num
        else:
            return {"error" : f'Unsupported operation {operation}'}
    
    except Exception as e:
        return {"error" : str(e)}



In [5]:
tools = [search_tool , calculator]
llm_with_tools = llm.bind_tools(tools)

In [6]:
class ChatState(TypedDict):
    messages : Annotated[list[BaseMessage] , add_messages]